In [56]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import nltk
import re
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rishikha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Import the dataset and rename the columns

In [57]:
df = pd.read_csv("C:\Rishikha\Projects\SMS_Spam_Detection\spam_sms.csv", encoding="latin-1")
df.columns = ['label', 'text']  # Rename columns

<>:1: SyntaxWarning: invalid escape sequence '\R'
<>:1: SyntaxWarning: invalid escape sequence '\R'
C:\Users\Rishikha\AppData\Local\Temp\ipykernel_14248\579632601.py:1: SyntaxWarning: invalid escape sequence '\R'
  df = pd.read_csv("C:\Rishikha\Projects\SMS_Spam_Detection\spam_sms.csv", encoding="latin-1")


In [58]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [59]:
print(df.shape)
print(df.columns)

(5572, 2)
Index(['label', 'text'], dtype='object')


Count the number of spam and ham messages

In [60]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [61]:
# Convert labels to 0 (ham) and 1 (spam)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [62]:
# Clean text function
def clean_text(text):
    text = text.lower()  # Coverted to lowercase
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text
df['text'] = df['text'].apply(clean_text)

In [63]:
# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(4457,)
(1115,)
(4457,)
(1115,)


In [64]:
max_vocab_size = 10000  # Vocabulary size
max_length = 100  # Sequence length

# Tokenize text
tokenizer = Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

# Pad sequences
x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad = pad_sequences(x_test_seq, maxlen=max_length, padding='post', truncating='post')

In [65]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)  # Increase spam examples
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_pad, y_train)

In [66]:
embedding_dim = 64  # Feature vector size

model = Sequential([
    Embedding(input_dim=max_vocab_size, output_dim=embedding_dim, input_length=max_length),  # Make sure these values are correctly set
    GRU(64, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

c:\Rishikha\Projects\SMS_Spam_Detection\venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [67]:
# Build model before calling summary
model.build(input_shape=(None, max_length))
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 665,025 (2.54 MB)

 Trainable params: 665,025 (2.54 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
history = model.fit(x_train_pad, y_train, epochs=10, batch_size=32, validation_data=(x_test_pad, y_test))

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.8658 - loss: 0.4126 - val_accuracy: 0.8655 - val_loss: 0.4016
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.8661 - loss: 0.3954 - val_accuracy: 0.8655 - val_loss: 0.3994
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.8661 - loss: 0.3987 - val_accuracy: 0.8655 - val_loss: 0.3972
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.8661 - loss: 0.3975 - val_accuracy: 0.8655 - val_loss: 0.4000
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.8661 - loss: 0.3966 - val_accuracy: 0.8655 - val_loss: 0.3962
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.8661 - loss: 0.3983 - val_accuracy: 0.8655 - val_loss: 0.4017
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 57ms/step - accuracy: 0.8661 - loss: 0.4019 - val_accuracy: 0.8655 - val_loss: 0.3991
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.8661 - loss: 0.3961 - val_acc

In [69]:
loss, accuracy = model.evaluate(x_test_pad, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8655 - loss: 0.3960
Test Accuracy: 86.55%


In [70]:
from sklearn.metrics import classification_report
y_pred = (model.predict(x_test_pad) > 0.5).astype("int32")
print(classification_report(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       965
           1       0.00      0.00      0.00       150

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



c:\Rishikha\Projects\SMS_Spam_Detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Rishikha\Projects\SMS_Spam_Detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Rishikha\Projects\SMS_Spam_Detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [71]:
def predict_spam(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad_seq = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')
    prediction = model.predict(pad_seq)[0][0]
    return "Spam" if prediction > 0.3 else "Not Spam"

In [72]:
message = input("Enter a message: ")
print(predict_spam(message))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Not Spam
